In [ ]:
# Dirichlet‑Process clustering of NDVI time series
# Piece‑wise (Loredo‑style) temporal model
# PyMC implementation with truncated DP and sequential updating

import numpy as np
import pymc as pm
import pytensor.tensor as pt
from matplotlib import pyplot as plt

import scipy.stats as stats

In [ ]:
# ============================================================
# USER PARAMETERS
# ============================================================
MAX_CLUSTERS = 5          # truncation level of DP
N_SEGMENTS   = 23          # piece‑wise bins for temporal model (e.g. 23 for 16‑day NDVI)
ALPHA_DP     = 3.0 #1.0         # DP concentration
SIGMA_MODIS  = 0.02        # NDVI noise (~MODIS typical)

In [ ]:
# ============================================================
# UTILITIES
# ============================================================

def doy_to_bin(doy, n_segments=N_SEGMENTS):
    """Map day-of-year to piecewise bin"""
    return np.floor((doy % 365) / 365 * n_segments).astype(int)

In [ ]:
def build_model_multi(series_bins, series_values):
    """
    series_bins : list of arrays of bins per series
    series_values: list of NDVI arrays
    """

    n_series = len(series_values)

    with pm.Model() as model:
        # ---------- Stick‑breaking DP ----------
        beta = pm.Beta("beta", 1, ALPHA_DP, shape=MAX_CLUSTERS)
        pi = pm.Deterministic("pi", pm.math.concatenate([beta[0:1], beta[1:] * pt.extra_ops.cumprod(1 - beta[:-1])]))
        pi = pi / pm.math.sum(pi)

        # ---------- Piecewise NDVI per cluster ----------
        a = pm.Uniform("a", lower=0, upper=1.0, shape=(MAX_CLUSTERS, N_SEGMENTS))
        # sigma = pm.HalfNormal("sigma", SIGMA_MODIS)

        # ---------- Cluster assignment per SERIES ----------
        z = pm.Categorical("z", p=pi, shape=n_series)

        # ---------- Likelihood per series ----------
        for i in range(n_series):
            mu_i = a[z[i], series_bins[i]]
            # pm.Normal(f"y_{i}", mu=mu_i, sigma=sigma,
            pm.Normal(f"y_{i}", mu=mu_i, sigma=SIGMA_MODIS,
            observed=series_values[i])

    return model


In [ ]:
def build_model_multi_hierar(series_bins, series_values):
    """
    series_bins : list of arrays of bins per series
    series_values: list of NDVI arrays
    """

    n_series = len(series_values)

    with pm.Model() as model:
        # ---------- Stick‑breaking DP ----------
        beta = pm.Beta("beta", 1, ALPHA_DP, shape=MAX_CLUSTERS)
        pi = pm.Deterministic("pi", pm.math.concatenate([beta[0:1], beta[1:] * pt.extra_ops.cumprod(1 - beta[:-1])]))
        pi = pi / pm.math.sum(pi)

        # ---------- Piecewise NDVI per cluster ----------
        # version uniforme
        # a = pm.Uniform("a", lower=0, upper=1.0, shape=(MAX_CLUSTERS, N_SEGMENTS))

        # version jerárquica
        # mu0 = pm.Normal("mu0", 0.4, 0.2, shape=N_SEGMENTS)
        # tau = pm.HalfNormal("tau", 0.1)

        # version jerárquica autotune
        # global_mean = series_values.mean()
        # global_std  = series_values.std()
        # mu0 = pm.Normal("mu0", global_mean, global_std, shape=N_SEGMENTS)
        # tau = pm.HalfNormal("tau", global_std/2)

        # Hiperprior jerárquico completo
        mu_global = pm.Normal("mu_global", 0.3, 0.5)
        sigma_global = pm.HalfNormal("sigma_global", 0.3)

        mu0 = pm.Normal("mu0", mu_global, sigma_global, shape=N_SEGMENTS)

        tau_scale = pm.HalfNormal("tau_scale", 0.3)
        tau = pm.HalfNormal("tau", tau_scale)

        a = pm.Normal("a", mu0, tau, shape=(MAX_CLUSTERS,N_SEGMENTS))
        # sigma = pm.HalfNormal("sigma", SIGMA_MODIS)

        # ---------- Cluster assignment per SERIES ----------
        z = pm.Categorical("z", p=pi, shape=n_series)

        # ---------- Likelihood per series ----------
        for i in range(n_series):
            mu_i = a[z[i], series_bins[i]]
            # pm.Normal(f"y_{i}", mu=mu_i, sigma=sigma,
            pm.Normal(f"y_{i}", mu=mu_i, sigma=SIGMA_MODIS,
            observed=series_values[i])

    return model


In [ ]:
def synthetic_series(n_series, anios, sigma=SIGMA_MODIS):
    """
    Generate NDVI synthetic series in MODIS 16-day composites (~23 per year)
    Clusters:
        0 → bosque perenne
        1 → cultivo anual
        2 → doble cultivo
    """

    n_quincenas = 23
    q = np.arange(n_quincenas*anios)

    series = []
    true_clusters = []

    for i in range(n_series):
        c = np.random.choice([0,1,2])
        true_clusters.append(c)

        if c == 0:   # bosque
            ndvi = 0.4 + 0.15*np.sin(2*np.pi*q/n_quincenas)

        elif c == 1: # cultivo anual
            ndvi = 0.2 + 0.5*((q%n_quincenas>6)&(q%n_quincenas<15))

        else:        # doble cultivo
            ndvi = 0.2 \
                 + 0.35*((q%n_quincenas>3)&(q%n_quincenas<7)) \
                 + 0.35*((q%n_quincenas>12)&(q%n_quincenas<16))

        ndvi = ndvi.astype(float)
        ndvi += np.random.normal(0, sigma, size=len(q))

        # clip NDVI realista
        ndvi = np.clip(ndvi, 0, 0.9)

        series.append(ndvi)

    return q, np.array(series), np.array(true_clusters)



In [ ]:
quincenas, series, true_clusters = synthetic_series(n_series=30, anios=3, sigma=SIGMA_MODIS)

In [ ]:
fig = plt.figure(figsize=(12,4))

plt.plot(quincenas, series[true_clusters == 0, :].T, color='green', alpha=0.5)
plt.plot(quincenas, series[true_clusters == 1, :].T, color='red', alpha=0.5)
plt.plot(quincenas, series[true_clusters == 2, :].T, color='blue', alpha=0.5)
plt.xlabel("Quincena")
plt.ylabel("NDVI")
plt.title("Series sintéticas con clusters conocidos")
# plt.plot(true_clusters)

### version avanzada inicializacion de clusters

In [ ]:
from scipy.stats import norm

def crp_init(series_values, alpha, max_clusters, sigma, n_segments):
    """
    Inicialización estilo Chinese Restaurant Process
    compatible con DP piece-wise
    """

    n_series = len(series_values)
    order = np.random.permutation(n_series)

    z = -np.ones(n_series, dtype=int)
    clusters = []

    for i in order:
        s = series_values[i]

        probs = []

        # --- clusters existentes ---
        for k, members in enumerate(clusters):
            nk = len(members)

            mean_curve = np.mean([series_values[j] for j in members], axis=0)

            ll = np.sum(norm.logpdf(s, mean_curve, sigma))
            probs.append(np.log(nk) + ll)

        # --- cluster nuevo ---
        ll_new = np.sum(norm.logpdf(s, np.mean(s), sigma))
        probs.append(np.log(alpha) + ll_new)

        probs = np.exp(probs - np.max(probs))
        probs /= probs.sum()

        k_new = np.random.choice(len(probs), p=probs)

        if k_new == len(clusters):
            clusters.append([i])
            z[i] = k_new
        else:
            clusters[k_new].append(i)
            z[i] = k_new

        if len(clusters) >= max_clusters:
            break

    # llenar no asignados
    z[z < 0] = 0
    return z

### Inferencia

In [ ]:
np.random.seed(0)

z_init = crp_init(
    series_values=series,
    alpha=ALPHA_DP,
    max_clusters=MAX_CLUSTERS,
    sigma=SIGMA_MODIS,
    n_segments=N_SEGMENTS
)

# quincenas, series, true_clusters = synthetic_series(n_series=30, anios=3)
# series_bins = [doy_to_bin(quincenas) for _ in series]
# series_bins = [quincenas for _ in series]
series_bins = [np.arange(len(s)) % 23 for s in series]
# model = build_model_multi(series_bins, series)
model = build_model_multi_hierar(series_bins, series)

# with model:
#     trace = pm.sample(draws=5000, tune=1000, chains=4)
#     z_post = trace.posterior["z"].mean(dim=("chain","draw")).values
#     z_est = np.round(z_post).astype(int)

with model:
    trace = pm.sample(draws=3000, 
                      tune=500, 
                      chains=4, 
                      target_accept=0.9, 
                      initvals=[{"z": z_init} for _ in range(4)])

In [ ]:
# model.free_RVs

for rv in model.free_RVs:
    print(rv.name, rv.eval().shape)

In [ ]:
z_samples = trace.posterior["z"].stack(samples=("chain","draw")).values
z_est = stats.mode(z_samples, axis=1, keepdims=False).mode

print("True clusters:")
print(true_clusters)
print("Estimated clusters:")
print(z_est)
print("Estimated N:", len(np.unique(z_est)))

### Evaluacion convergencia

In [ ]:
trace.posterior

In [ ]:
import arviz as az
# az.summary(trace, var_names=["z"])
az.summary(trace)

In [ ]:
trace.posterior["pi"].mean(dim=("chain","draw")).values

### Evaluacion clusters

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

def aligned_confusion(true_labels, est_labels):
    true_labels = np.array(true_labels)
    est_labels  = np.array(est_labels)

    cm = confusion_matrix(true_labels, est_labels)

    # Hungarian algorithm → maximizar diagonal
    row_ind, col_ind = linear_sum_assignment(-cm)

    aligned_cm = cm[:, col_ind]
    mapping = {col_ind[i]: row_ind[i] for i in range(len(row_ind))}

    aligned_est = np.array([mapping[e] for e in est_labels])

    return aligned_cm, aligned_est, mapping


In [ ]:
def plot_confusion(cm, title="Confusion Matrix"):
    plt.figure(figsize=(6,5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.xlabel("Estimated cluster")
    plt.ylabel("True cluster")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i,j],
                     ha="center", va="center",
                     fontsize=12)

    plt.tight_layout()
    plt.show()


In [ ]:
uniques = np.unique(z_est)

In [ ]:
z_est[z_est == 4] = 2

In [ ]:
cm, aligned_est, mapping = aligned_confusion(true_clusters, z_est)

print("Cluster mapping:", mapping)
plot_confusion(cm)


Evaluacion series de tiempo

In [ ]:
def plot_cluster_ndvi_posterior(trace, quincenas, z_est, ci=0.9):
    """
    trace   : trace de PyMC
    quincenas    : array quincenas
    z_est   : cluster asignado por serie
    """

    a_post = trace.posterior["a"].stack(sample=("chain","draw")).values
    # shape = (Kmax, N_segments, n_samples)

    clusters = np.unique(z_est)
    # bins = np.floor((days % 365) / 365 * n_segments).astype(int)
    bins = quincenas

    plt.figure(figsize=(10,4))
    
    for k in clusters:

        samples = a_post[k, bins, :]  # (365, n_samples)

        mean = samples.mean(axis=1)
        low  = np.percentile(samples, (1-ci)/2*100, axis=1)
        high = np.percentile(samples, (1+(ci))/2*100, axis=1)

        n_series = np.sum(z_est == k)

        plt.plot(quincenas, mean, label=f"Cluster {k} (n={n_series})")
        plt.fill_between(quincenas, low, high, alpha=0.25)

    # series reales
    plt.plot(quincenas[:23], series[true_clusters == 0, :].T[:23], color='green', alpha=0.1)
    plt.plot(quincenas[:23], series[true_clusters == 1, :].T[:23], color='red', alpha=0.1)
    plt.plot(quincenas[:23], series[true_clusters == 2, :].T[:23], color='blue', alpha=0.1)
    plt.xlabel("Quincena")
    plt.ylabel("NDVI")
    plt.title("NDVI promedio por cluster (posterior PyMC)")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plot_cluster_ndvi_posterior(trace, quincenas[:23], z_est, ci=0.9)